# LLM → SPICE → GDS
SSCS Chipathon 2026 — gLayout Track

Describe an analog block in natural language, have an LLM write the SPICE,
then generate a DRC-clean GDSII layout and verify it.

**PDK**: gf180mcuD  |  **Supply**: 3.3 V

Runs without an API key — it falls back to a reference netlist.


In [ ]:
# ── Bootstrap ────────────────────────────────────────────────────────
import os, sys

try:
    import mbg
except ModuleNotFoundError:
    d = os.path.abspath("")
    for _ in range(8):
        if os.path.isdir(os.path.join(d, "src", "mbg")):
            sys.path.insert(0, os.path.join(d, "src")); break
        d = os.path.dirname(d)
    import mbg

os.environ.setdefault("PDK_ROOT", os.path.expanduser("~/.volare"))
os.environ.setdefault("PDK", "gf180mcuD")
os.environ.setdefault("PDKPATH", os.path.join(os.environ["PDK_ROOT"], os.environ["PDK"]))
os.environ.setdefault("STD_CELL_LIBRARY", "gf180mcu_fd_sc_mcu7t5v0")
PDK_LIB = os.path.join(os.environ["PDKPATH"], "libs.tech", "ngspice", "sm141064.ngspice")
print("mbg", mbg.__version__, "| PDK", os.environ["PDK"])


In [ ]:
# ── 1. LLM credentials (optional) ────────────────────────────────────
# Set DEEPSEEK_API_KEY in the environment or a .env file.
# Without a key the notebook falls back to a built-in reference netlist, so
# every later cell still runs.
from mbg.llm import _load_api_key

# Set USE_LLM = True to actually call the model. It is off by default so this
# notebook is reproducible: an LLM writes a different netlist every run, and a
# notebook that changes its own result is not something you can test.
USE_LLM = os.environ.get("MBG_USE_LLM", "").lower() in ("1", "true", "yes")

api_key = _load_api_key()
HAVE_LLM = USE_LLM and bool(api_key)

if not USE_LLM:
    print("LLM disabled (set MBG_USE_LLM=1 to enable) — using the reference netlist")
elif not api_key:
    print("MBG_USE_LLM set but DEEPSEEK_API_KEY missing — using the reference netlist")
else:
    print(f"LLM enabled, key loaded ({api_key[:8]}...)")


In [ ]:
# ── 2. Describe the circuit, and let the LLM write the SPICE ─────────
from mbg import generate_netlist_from_prompt

prompt = """Design a 5-transistor OTA for the GF180MCU 3.3V PDK.
NMOS differential input pair, PMOS current-mirror load, NMOS tail current source.
Use only nfet_03v3 and pfet_03v3. Subcircuit name: ota_5t.
Ports: vdd vss inp inm out vb"""

FALLBACK = f""".lib "{PDK_LIB}" typical
.subckt ota_5t vdd vss inp inm out vb
XM1  net1 inp net2 vss nfet_03v3 L=1u W=4u nf=4
XM2  out  inm net2 vss nfet_03v3 L=1u W=4u nf=4
XM3  net1 net1 vdd vdd pfet_03v3 L=1u W=4u nf=4
XM4  out  net1 vdd vdd pfet_03v3 L=1u W=4u nf=4
XM5  net2 vb  vss vss nfet_03v3 L=1u W=4u nf=4
.ends
"""

netlist = None
if HAVE_LLM:
    netlist = generate_netlist_from_prompt(prompt)
    if not netlist:
        print("LLM call failed — falling back")
if not netlist:
    netlist = FALLBACK

print(netlist)


In [ ]:
# ── 3. Inspect what the netlist describes ────────────────────────────
# Constraint extraction finds differential pairs and current mirrors with no
# annotation in the netlist.
from mbg import parse_netlist_with_pdk, build_design_context
from glayout import gf180

ctx = build_design_context(parse_netlist_with_pdk(netlist), gf180)

print(f"  devices : {len(ctx.devices)}")
print(f"  nets    : {len(ctx.nets)}")
for g in ctx.matching_groups.values():
    print(f"  {g.kind:16s} {g.devices}")
print(f"  sensitive nets : {sorted(ctx.sensitive_nets)}")
print(f"  critical nets  : {sorted(ctx.critical_nets)}")


In [ ]:
# ── 4. Pre-layout simulation (ngspice) ───────────────────────────────
# run_spice executes exactly the deck it is given, so the deck must carry its
# own analysis statements. GF180 models also need design.ngspice included
# BEFORE the model library, otherwise ngspice reports "Formula() error".
import shutil
from mbg import run_spice

DESIGN_INC = os.path.join(os.environ["PDKPATH"], "libs.tech", "ngspice", "design.ngspice")

if shutil.which("ngspice") is None:
    print("ngspice not on PATH — skipping simulation")
elif not os.path.isfile(DESIGN_INC):
    print(f"{DESIGN_INC} not found — skipping simulation")
else:
    core = "\n".join(l for l in netlist.splitlines() if not l.strip().lower().startswith(".lib"))
    tb = f""".include '{DESIGN_INC}'
.lib '{PDK_LIB}' typical
{core}

Vvdd vdd 0 3.3
Vvss vss 0 0
Vb   vb  0 0.8
Vinp inp 0 1.65
Vinm inm 0 1.65
X1 vdd vss inp inm out vb ota_5t
CL out vss 1p

.control
op
print v(out)
.endc
.end
"""
    res = run_spice(tb, workdir="sim_out", fmt="raw")

    # ngspice frequently exits non-zero even when the analysis succeeded, so
    # judge the run by whether it produced a value rather than by the code.
    import re
    m = re.search(r"v\(out\)\s*=\s*([-\d.eE+]+)", res["stdout"] or "")
    if m:
        print(f"  operating point: v(out) = {float(m.group(1)):.4f} V")
        print(f"  (ngspice exit code {res['returncode']} — not meaningful after a .control block)")
    else:
        print("  simulation produced no v(out); ngspice said:")
        for line in (res["stderr"] or res["stdout"] or "").splitlines()[-8:]:
            print("   ", line)


In [ ]:
# ── 5. SPICE → GDS + DRC + LVS + PEX ─────────────────────────────────
from mbg import spice_to_gds_with_checks

r = spice_to_gds_with_checks(netlist)

print()
print(f"  GDS : {r['gds_path']}")
print(f"  DRC : {r['drc']['summary']}")
print(f"  LVS : {'MATCH' if r['lvs']['match'] else 'MISMATCH'}")
print(f"  ALL : {'PASS' if r['all_pass'] else 'FAIL'}")

from IPython.display import SVG, display
if r.get("svg_path") and os.path.isfile(r["svg_path"]):
    display(SVG(r["svg_path"]))
